# 04 Descriptive statistics and figures  (Step 0, feature engineering)

Describes the dataset the models are trained on: composition by city and split, height distributions, sampling quota,
spatial layout, distances between test and training points, the structure and redundancy of the 1,872 features, and the
feature extraction scheme figure. Nothing is fitted or learned here.

| Section | Needs | Output (`outputs/`) |
|---|---|---|
| 1. Dataset overview | the shipped features table only | `dataset_overview/`: figures `fig1a` to `fig7` and CSV summaries |
| 2. Sentinel-2 chip (optional) | Earth Engine account, `pip install earthengine-api requests` | `figures/s2_rgb_chip.tif` |
| 3. Feature extraction scheme | the footprints of Sao Paulo (`data_raw/`) and the chip from section 2; `DEMO = True` draws the layout on synthetic imagery without any data | `figures/feature_extraction_scheme.png/.pdf` |

In [ ]:
# Setup: make `config.py` importable whether Jupyter was started in the repo root or in notebooks/
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, _here / "notebooks", _here.parent / "notebooks"):
    if (_p / "config.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise FileNotFoundError("config.py not found: start Jupyter from the repository root or from notebooks/")
import config
print("repo root:", config.ROOT)

## 1. Dataset overview
Figures: `fig1a_composition` (rows per city and split), `fig1b_class_shares` (height classes, train vs natural test),
`fig2_height_distribution`, `fig3_quota_per_slice`, `fig4_spatial_layout`, `fig5_distance_to_train`,
`fig6_feature_structure` (18 layers x 13 statistics x 8 variants, missing values) and `fig7_feature_redundancy`;
tables: `summary_by_city_split.csv`, `slice_counts.csv`, `feature_structure.csv`, `missing_by_layer_variant.csv`, `distance_summary.csv`.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")

# ------------------------- CONFIG ------------------------------------

TABLE = str(config.TABLE)                  # the shipped features table (only the `split` column and the features are used)
OUT = str(config.out("dataset_overview"))
QUOTA = None           # the first sampling used 200 per 1 m slice and city; the rebalanced set has no fixed quota (None = no line)
TOP_SLICE = 49         # last slice [49 m, inf) pools the rare high-rises
CORR_ROWS = 4000       # training rows used for the correlation matrix
SEED = 42
# ---------------------------------------------------------------------

META = ["sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split"]
CLASS_EDGES = [2, 6, 15, 30]
CLASS_NAMES = ["<2 m (background)", "2-6 m", "6-15 m", "15-30 m", ">30 m"]
LAYERS = ["VV", "VH", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12",
          "NDVI", "TCG", "NDBI", "TCB", "TCW", "mNDWI"]
STATS = ["mean", "stdDev", "p0", "p10", "p25", "p50", "p75", "p90", "p100", "range", "IQR", "skew", "kurtosis"]
VARIANTS = ["BASE", "ERO", "DIL", "OPN", "CLS", "GRD", "THT", "BHT"]
CITY_COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]   # fixed order, one hue per city (8 slots)
TRAIN_C, TEST_C, EXCL_C = "#9a9992", "#3a3935", "#e34948"
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def _style(ax, grid=True):
    ax.set_facecolor(SURF)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9)
    if grid:
        ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True)


def _grid(n, w, h, sharey=True, ncols=4):
    """One panel per city in a grid of at most `ncols` columns (8 cities in a single row would be 35 inches wide)."""
    nc = min(n, ncols); nr = int(np.ceil(n / nc))
    fig, axes = plt.subplots(nr, nc, figsize=(w * nc, h * nr), sharey=sharey, squeeze=False)
    flat = axes.ravel()
    for ax in flat[n:]:
        ax.set_visible(False)
    if nr > 1:
        fig.subplots_adjust(hspace=0.55)
    return fig, flat[:n]


def _save(fig, name):
    fig.savefig(f"{OUT}/{name}.png", dpi=160, bbox_inches="tight", facecolor=SURF)
    plt.show()


def parse_feature(name):
    t = name.split("_")
    if len(t) == 3 and t[2] in VARIANTS:
        return t[0], t[1], t[2]
    if len(t) == 2:
        return t[0], t[1], "BASE"
    return None


def class_of(h):
    return np.digitize(h, CLASS_EDGES)


# ------------------- main ----------------------------------------------
def dataset_overview():
    os.makedirs(OUT, exist_ok=True)
    plt.rcParams.update({"figure.facecolor": SURF, "axes.labelcolor": INK2, "text.color": INK,
                         "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold"})
    df = config.read_table()
    feats = [c for c in df.columns if c not in META and c not in {"system:index", ".geo", "n_parts_50m", "slice_lo"}]
    df[feats] = df[feats].apply(pd.to_numeric, errors="coerce").astype("float32")
    cities = sorted(df.city.unique())
    col = {c: CITY_COL[i % len(CITY_COL)] for i, c in enumerate(cities)}
    splits = [s for s in ["train", "test", "excluded"] if s in set(df.split)]
    bld = df[df.is_background == 0]
    print(f"{len(df)} rows, {len(feats)} features, cities {cities}")

    # ---- 1a composition ------------------------------------------------
    splits_all = splits + (["unused"] if "unused" in set(df.split) else [])   # unused = train rows left out by the rebalancing
    comp = df.groupby(["city", "split"]).size().unstack(fill_value=0).reindex(columns=splits_all, fill_value=0)
    comp.loc["ALL"] = comp.sum()
    print("\nRows per city and split:\n", comp.to_string())
    fig, ax = plt.subplots(figsize=(max(7.5, 1.15 * len(cities) + 2), 4.4)); _style(ax); ax.grid(axis="x", visible=False)
    xs = np.arange(len(cities)); bottom = np.zeros(len(cities))
    for s, c in zip(splits_all, [TRAIN_C, TEST_C, EXCL_C, "#e6e5e1"]):
        v = comp.loc[cities, s].to_numpy()
        ax.bar(xs, v, bottom=bottom, color=c, edgecolor=SURF, linewidth=2, width=0.6, label=s)
        for x, b, n in zip(xs, bottom, v):
            if n < 0.045 * comp.loc[cities].sum(axis=1).max():
                continue          # segment too thin for a label
            ax.text(x, b + n / 2, f"{n:,}", ha="center", va="center", fontsize=9, color="white" if s in ("test", "excluded") else INK)
        bottom += v
    ax.set_xticks(xs); ax.set_xticklabels(cities, rotation=25, ha="right"); ax.set_ylabel("rows")
    ax.set_title("Rows per city and split", loc="left"); ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5))
    _save(fig, "fig1a_composition")

    # ---- 1b class shares ----------------------------------------------
    fig, axes = _grid(len(cities), 4.4, 3.8)
    axes = np.atleast_1d(axes); rows = []
    for ax, c in zip(axes, cities):
        _style(ax); ax.grid(axis="x", visible=False)
        for k, (s, colr, hatch, off) in enumerate([("train", TRAIN_C, None, -0.2), ("test", TEST_C, "///", 0.2)]):
            if s not in splits:
                continue
            h = df.loc[(df.city == c) & (df.split == s), "height_m"].to_numpy()
            sh = np.bincount(class_of(h), minlength=5) / max(len(h), 1)
            ax.bar(np.arange(5) + off, sh, 0.38, color=colr, edgecolor=SURF, linewidth=2, hatch=hatch, label=s)
            rows += [dict(city=c, split=s, cls=n, share=v) for n, v in zip(CLASS_NAMES, sh)]
        ax.set_xticks(range(5)); ax.set_xticklabels(["<2", "2-6", "6-15", "15-30", ">30"])
        ax.set_xlabel("height class (m)"); ax.set_title(c, loc="left")
    axes[0].set_ylabel("share of rows"); axes[0].legend(frameon=False)
    fig.suptitle("Height classes: stratified training sample vs natural test sample", x=0.01, y=1.04, ha="left", fontsize=11, color=INK2)
    _save(fig, "fig1b_class_shares")
    pd.DataFrame(rows).round(4).to_csv(f"{OUT}/class_shares.csv", index=False)

    # ---- 2 height distribution ----------------------------------------
    edges = np.r_[np.arange(3, 61, 1.0), np.inf]
    fig, axes = _grid(len(cities), 4.8, 3.9)
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, cities):
        _style(ax)
        for s, colr in [("train", TRAIN_C), ("test", TEST_C)]:
            if s not in splits:
                continue
            h = bld.loc[(bld.city == c) & (bld.split == s), "height_m"].to_numpy()
            cnt = np.histogram(np.clip(h, 0, 60.5), bins=np.r_[edges[:-1], 61.0])[0] / max(len(h), 1)
            if s == "train":
                ax.bar(edges[:-1] + 0.5, cnt, 1.0, color=colr, edgecolor=SURF, linewidth=0.8, label="train (stratified)")
            else:
                ax.step(np.r_[edges[:-1], 61.0], np.r_[cnt, cnt[-1]], where="post", color=INK, lw=1.8, label="test (natural)")
        nb = ", ".join(f"{s} {int(((df.city == c) & (df.split == s) & (df.is_background == 1)).sum())}" for s in splits)
        mx = df.loc[(df.city == c) & (df.is_background == 0), "height_m"].max()
        ax.text(0.97, 0.97, f"background rows: {nb}\nmax label {mx:.0f} m", transform=ax.transAxes,
                ha="right", va="top", fontsize=8.5, color=INK2)
        ax.set_xlabel("label height (m), buildings only; last bar = 60+"); ax.set_title(c, loc="left")
    axes[0].set_ylabel("share of building rows per metre"); axes[-1].legend(frameon=False, fontsize=9, loc="center right")
    _save(fig, "fig2_height_distribution")

    # ---- 3 quota per slice --------------------------------------------
    samp = bld[bld.split.isin(["train", "excluded"])].copy()
    samp["slice"] = np.minimum(np.floor(samp.height_m), TOP_SLICE).astype(int)
    sl = samp.groupby(["city", "slice"]).size().unstack(0).reindex(range(3, TOP_SLICE + 1)).fillna(0)
    sl.to_csv(f"{OUT}/slice_counts.csv")
    fig, ax = plt.subplots(figsize=(9.5, 4.2)); _style(ax)
    for c in cities:
        ax.plot(sl.index, sl[c], color=col[c], marker="o", ms=3.5, lw=1.8, label=c)
    if QUOTA:
        ax.axhline(QUOTA, color=INK2, lw=1, ls=(0, (6, 3)))
        ax.text(sl.index.max(), QUOTA + 5, f"quota {QUOTA} per slice", ha="right", fontsize=9, color=INK2)
    tk = list(range(3, TOP_SLICE - 3, 5)) + [TOP_SLICE]; ax.set_xticks(tk); ax.set_xticklabels([str(i) for i in tk[:-1]] + [f"{TOP_SLICE}+"])
    ax.set_xlabel("1 m height slice (m); last slice pools everything from 49 m"); ax.set_ylabel("sampled rows in the slice")
    ax.set_title("Rows per 1 m slice in the training set (train + excluded)", loc="left"); ax.legend(frameon=False)
    _save(fig, "fig3_quota_per_slice")
    if QUOTA:
        short = {c: int((sl[c] < QUOTA * 0.95).sum()) for c in cities}
        print(f"\nSlices below 95% of the quota (of {len(sl)}): {short}")

    # ---- 4 spatial layout ---------------------------------------------
    fig, axes = _grid(len(cities), 5, 4.6, sharey=False)
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, cities):
        _style(ax, grid=False)
        g = df[df.city == c]; x0, y0 = g.x_m.min(), g.y_m.min()
        for s, colr, mk, sz in [("train", TRAIN_C, "o", 4), ("test", TEST_C, "o", 5), ("excluded", EXCL_C, "x", 14)]:
            if s in splits:
                q = g[g.split == s]
                ax.scatter((q.x_m - x0) / 1000, (q.y_m - y0) / 1000, s=sz, c=colr, marker=mk, linewidths=0.8,
                           alpha=0.8 if s != "train" else 0.55, label=f"{s} ({len(q):,})")
        ax.set_aspect("equal"); ax.set_xlabel("km east of the south-west corner"); ax.set_title(c, loc="left")
        ax.legend(frameon=False, fontsize=8, markerscale=1.6, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3)
    axes[0].set_ylabel("km north")
    _save(fig, "fig4_spatial_layout")

    # ---- 5 distances ---------------------------------------------------
    fig, axes = _grid(len(cities), 4.8, 3.8)
    axes = np.atleast_1d(axes); drows = []
    for ax, c in zip(axes, cities):
        _style(ax)
        tr = df[(df.city == c) & (df.split == "train")][["x_m", "y_m"]].to_numpy(float)
        te = df[(df.city == c) & (df.split == "test")][["x_m", "y_m"]].to_numpy(float)
        if len(tr) < 2 or len(te) < 1:
            continue
        tree = cKDTree(tr)
        d_te = tree.query(te)[0]; d_tr = tree.query(tr, k=2)[0][:, 1]
        bins = np.logspace(np.log10(10), np.log10(max(d_te.max(), d_tr.max()) * 1.05), 40)
        ax.hist(d_tr, bins=bins, histtype="step", color=TRAIN_C, lw=2, label="train to nearest other train")
        ax.hist(d_te, bins=bins, histtype="step", color=INK, lw=2, label="test to nearest train")
        for v in (50, 110, 330):
            ax.axvline(v, color=INK2, lw=0.8, ls=(0, (4, 3)))
        ax.set_xscale("log"); ax.set_xlabel("distance (m); dashed: 50, 110, 330 m"); ax.set_title(c, loc="left")
        drows.append(dict(city=c, min_test_to_train_m=d_te.min(), median_test_to_train_m=np.median(d_te),
                          share_test_within_110m=(d_te < 110).mean(), share_test_within_330m=(d_te < 330).mean(),
                          min_train_to_train_m=d_tr.min(), median_train_to_train_m=np.median(d_tr)))
    axes[0].set_ylabel("points"); axes[0].legend(frameon=True, facecolor=SURF, edgecolor="none", framealpha=0.95, fontsize=8, loc="upper left")
    _save(fig, "fig5_distance_to_train")
    dsum = pd.DataFrame(drows).round(3); dsum.to_csv(f"{OUT}/distance_summary.csv", index=False)
    print("\nNearest-neighbour distances:\n", dsum.to_string(index=False))

    # ---- 6 feature structure and missing values -------------------------
    parsed = {f: parse_feature(f) for f in feats}
    bad = [f for f, p in parsed.items() if p is None]
    fs = pd.DataFrame([(f, *p) for f, p in parsed.items() if p], columns=["feature", "layer", "stat", "variant"])
    print(f"\nFeature table: {len(feats)} columns, {len(fs)} parsed as layer_stat[_variant]; unparsed: {bad[:5]}")
    print(f"layers {fs.layer.nunique()} x stats {fs.stat.nunique()} x variants {fs.variant.nunique()} "
          f"= {fs.layer.nunique() * fs.stat.nunique() * fs.variant.nunique()} (table has {len(fs)})")
    fs.to_csv(f"{OUT}/feature_structure.csv", index=False)
    nan_frac = df[list(fs.feature)].isna().mean()
    fs["nan_frac"] = fs.feature.map(nan_frac)
    heat = fs.pivot_table(index="layer", columns="variant", values="nan_frac", aggfunc="mean") \
             .reindex(index=[l for l in LAYERS if l in set(fs.layer)], columns=[v for v in VARIANTS if v in set(fs.variant)]) * 100
    heat.round(3).to_csv(f"{OUT}/missing_by_layer_variant.csv")
    rows_nan = df[list(fs.feature)].isna().any(axis=1)
    print(f"Missing values: {int(df[list(fs.feature)].isna().sum().sum()):,} cells "
          f"({df[list(fs.feature)].isna().mean().mean() * 100:.3f}% of all cells); "
          f"{int(rows_nan.sum())} rows ({rows_nan.mean() * 100:.2f}%) have at least one; "
          f"{int((nan_frac > 0).sum())} features have at least one")
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 5.6), gridspec_kw={"width_ratios": [1.25, 1]})
    _style(a0, grid=False)
    a0.axis("off")
    txt = ("Features = 18 layers x 13 statistics x 8 variants\n\n"
           "Layers (18)\n  VV, VH\n  S2 bands B2 B3 B4 B5 B6 B7 B8 B8A B11 B12\n  indices NDVI, TCG, NDBI, TCB, TCW, mNDWI\n\n"
           "Statistics (13) over the time window\n  mean, stdDev, p0, p10, p25, p50, p75, p90, p100,\n  range, IQR, skew, kurtosis\n  (234 spectral-temporal metrics)\n\n"
           "Variants (8), 50 m circular element\n  BASE = the metric itself\n  ERO erosion, DIL dilation, OPN opening,\n  CLS closing, GRD gradient, THT top-hat,\n  BHT black-hat\n\n"
           "Frantz et al. (2021): 234 x 7 = 1,638 features\nHere: 234 x 8 (adds BASE) = 1,872\n\n"
           f"Rows: {len(df):,}\nMissing cells: {df[list(fs.feature)].isna().mean().mean() * 100:.3f}%")
    a0.text(0.0, 1.0, txt, va="top", ha="left", fontsize=9, color=INK, linespacing=1.25)
    _style(a1, grid=False)
    im = a1.imshow(heat.to_numpy(), aspect="auto", cmap="Blues", vmin=0, vmax=max(float(heat.to_numpy().max()), 1.0))
    a1.set_xticks(range(heat.shape[1])); a1.set_xticklabels(heat.columns); a1.set_yticks(range(heat.shape[0])); a1.set_yticklabels(heat.index)
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            a1.text(j, i, f"{heat.iat[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color=INK)
    a1.set_title("Missing values (% of rows) per layer and variant", loc="left")
    fig.colorbar(im, ax=a1, fraction=0.04, pad=0.02, label="% missing")
    _save(fig, "fig6_feature_structure")

    # ---- 7 redundancy ---------------------------------------------------
    trn = df[df.split == "train"]
    samp_idx = np.random.default_rng(SEED).choice(len(trn), size=min(CORR_ROWS, len(trn)), replace=False)
    order = fs.assign(l=fs.layer.map({l: i for i, l in enumerate(LAYERS)}), v=fs.variant.map({v: i for i, v in enumerate(VARIANTS)}),
                      s=fs.stat.map({s: i for i, s in enumerate(STATS)})).sort_values(["l", "v", "s"]).feature.tolist()
    X = trn.iloc[samp_idx][order].to_numpy(np.float64)
    X = np.where(np.isnan(X), np.nanmedian(X, axis=0, keepdims=True), X)
    X = X[:, X.std(axis=0) > 0]
    C = np.abs(np.corrcoef(X, rowvar=False)); iu = np.triu_indices_from(C, k=1); v = C[iu]
    shares = {t: float((v > t).mean()) for t in (0.8, 0.9, 0.95)}
    print(f"\nFeature redundancy on {len(X)} training rows, {X.shape[1]} non-constant features: "
          + ", ".join(f"|r| > {t}: {s * 100:.1f}% of pairs" for t, s in shares.items()))
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.1, 1]})
    im = a0.imshow(C, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    a0.set_title("|correlation| between features (ordered by layer, variant, statistic)", loc="left", fontsize=10)
    a0.set_xticks([]); a0.set_yticks([]); fig.colorbar(im, ax=a0, fraction=0.04, pad=0.02)
    _style(a1)
    a1.hist(v, bins=50, color=TRAIN_C, edgecolor=SURF)
    a1.set_xlabel("|r| between two features"); a1.set_ylabel("feature pairs")
    a1.set_title(f"{shares[0.9] * 100:.0f}% of pairs have |r| > 0.9", loc="left")
    _save(fig, "fig7_feature_redundancy")

    # ---- summary table --------------------------------------------------
    out = []
    for (c, s), g in df.groupby(["city", "split"]):
        b = g[g.is_background == 0].height_m
        out.append(dict(city=c, split=s, rows=len(g), background=int(g.is_background.sum()), buildings=len(b),
                        mean_m=b.mean(), median_m=b.median(), p10_m=b.quantile(.1), p90_m=b.quantile(.9), max_m=b.max(),
                        share_3_to_6m=((b >= 3) & (b < 6)).mean(), share_above_15m=(b >= 15).mean(), share_above_30m=(b >= 30).mean(),
                        n_above_30m=int((b >= 30).sum())))
    summ = pd.DataFrame(out).round(3); summ.to_csv(f"{OUT}/summary_by_city_split.csv", index=False)
    print("\nHeight statistics (buildings only):\n", summ.to_string(index=False))
    print(f"\nFigures and tables written to {OUT}")
    return dict(summary=summ, composition=comp, slices=sl, distances=dsum, features=fs, missing=heat, redundancy=shares)


overview = dataset_overview()

## 2. Sentinel-2 natural-colour chip (optional, Earth Engine Python API)
Downloads a 500 m chip over central Sao Paulo straight to `outputs/figures/`. If you prefer the Code Editor, skip this section: section 3 prints a JavaScript snippet that exports the same chip to Drive.

In [ ]:
import os, io, zipfile
import numpy as np
import requests
import ee
from pyproj import Transformer

# ------------------------- CONFIG ------------------------------------
EE_PROJECT = "YOUR-EARTH-ENGINE-PROJECT"    # SET: Google Cloud project enabled for Earth Engine

OUT_TIF = os.path.join(str(config.out("figures")), "s2_rgb_chip.tif")
CENTER_LONLAT = (-46.6333, -23.5505)           # same value as in section 3
CHIP_HALF_M = 250
EPSG = 31983                                   # SIRGAS 2000 / UTM 23S (Sao Paulo)
DATE_START, DATE_END = "2019-01-01", "2021-01-01"   # the 2-year window used for the features
CS_THRESHOLD = 0.6                             # Cloud Score+ cs_cdf, as in the feature export
# ---------------------------------------------------------------------

try:
    ee.Initialize(project=EE_PROJECT)
except Exception as e:
    print("ee.Initialize failed (", type(e).__name__, "), authenticating once...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

cx, cy = Transformer.from_crs(4326, EPSG, always_xy=True).transform(*CENTER_LONLAT)
xmin, ymin = np.floor((cx - CHIP_HALF_M) / 10) * 10, np.floor((cy - CHIP_HALF_M) / 10) * 10
xmax, ymax = xmin + 2 * CHIP_HALF_M, ymin + 2 * CHIP_HALF_M
region = ee.Geometry.Rectangle([float(xmin), float(ymin), float(xmax), float(ymax)], f"EPSG:{EPSG}", False)

sr = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(region).filterDate(DATE_START, DATE_END)
cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED").filterBounds(region).filterDate(DATE_START, DATE_END)
clear = sr.linkCollection(cs, ["cs_cdf"]).map(lambda i: i.updateMask(i.select("cs_cdf").gte(CS_THRESHOLD)))
rgb = clear.select(["B4", "B3", "B2"]).median().visualize(min=0, max=3000, gamma=1.3)

print("S2 scenes in window:", sr.size().getInfo())
url = rgb.getDownloadURL({
    "region": region, "crs": f"EPSG:{EPSG}",
    "crs_transform": [10, 0, float(xmin), 0, -10, float(ymax)],
    "format": "GEO_TIFF",
})
r = requests.get(url, timeout=300); r.raise_for_status()
content = r.content
if content[:2] == b"PK":                       # some responses come zipped
    z = zipfile.ZipFile(io.BytesIO(content))
    names = [n for n in z.namelist() if n.lower().endswith((".tif", ".tiff"))]
    if len(names) == 1:
        content = z.read(names[0])
    else:                                      # one file per band: stack them
        import rasterio
        bands = {n.split(".")[-2]: z.read(n) for n in names}
        arrs = []
        for k in ("vis-red", "vis-green", "vis-blue"):
            with rasterio.MemoryFile(bands[k]) as mf, mf.open() as src:
                arrs.append(src.read(1)); prof = src.profile
        prof.update(count=3)
        os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)
        with rasterio.open(OUT_TIF, "w", **prof) as dst:
            for i, a in enumerate(arrs, 1):
                dst.write(a, i)
        content = None
if content is not None:
    os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)
    open(OUT_TIF, "wb").write(content)

import rasterio
with rasterio.open(OUT_TIF) as src:
    print("saved:", OUT_TIF, "| size", src.width, "x", src.height, "| bands", src.count, "| CRS", src.crs, "| bounds", src.bounds)
print("Chip ready: run section 3.")

## 3. Feature extraction scheme
One figure showing how a sample pixel becomes a row of tabular features (1,872) and a CNN patch (48 channels, 33 x 33 px).
The tabular features use morphological operators (Frantz et al. 2021), not texture; the seasonal composites belong to the CNN branch only.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch
from matplotlib.lines import Line2D

# ------------------------- CONFIG ------------------------------------

OUT = str(config.out("figures"))
DEMO = False                                   # True: synthetic placeholder imagery + footprints
FOOTPRINTS = str(config.FOOTPRINTS("sao_paulo"))   # the building footprints used by the sampler (data_raw/_footprints/)
HEIGHT_FIELD = "height_m"                      # SET: attribute with the part height (m)
RGB_TIF = os.path.join(OUT, "s2_rgb_chip.tif")      # chip written by section 2
if not os.path.exists(RGB_TIF):
    RGB_TIF = None                              # then the cell prints an Earth Engine Code Editor (JavaScript) snippet that exports the chip
CENTER_LONLAT = (-46.6333, -23.5505)           # window centre (lon, lat); here Praca da Se, Sao Paulo
CHIP_HALF_M = 250                              # half width of the chip exported from Earth Engine
PICK_XY = None                                 # optional [(x1, y1), (x2, y2)] building centroids (chip CRS, m)
MIN_AREA_M2 = 200                              # ignore tiny parts when picking the two examples
SEED = 3
# ---------------------------------------------------------------------

R_M, PIX = 50.0, 10.0
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#d9d8d3", "#fcfcfb"
BLUE, BLUE_D, ORANGE, RED, AQUA, MUTE = "#2a78d6", "#1d5aa6", "#eb6834", "#e34948", "#1baf7a", "#9a9992"
C30, S30 = np.cos(np.radians(30)), np.sin(np.radians(30))


# ------------------- data -----------------------------------------------
def demo_data():
    import geopandas as gpd
    from shapely.geometry import box
    rng = np.random.default_rng(SEED)
    x0, y0, W, H = 333300.0, 7394300.0, 500.0, 400.0     # near Sao Paulo downtown, UTM 23S
    rows = []
    street, blk = 14.0, 76.0
    for bx in np.arange(x0, x0 + W, blk + street):
        for by in np.arange(y0, y0 + H, blk + street):
            nx, ny = rng.integers(2, 4), rng.integers(2, 4)
            xs, ys = np.linspace(bx, bx + blk, nx + 1), np.linspace(by, by + blk, ny + 1)
            for i in range(nx):
                for j in range(ny):
                    gx, gy = rng.uniform(1.5, 3.5), rng.uniform(1.5, 3.5)
                    rows.append((box(xs[i] + gx, ys[j] + gy, xs[i + 1] - gx, ys[j + 1] - gy),
                                 float(np.clip(rng.gamma(2.0, 14.0), 4, 110))))
    gdf = gpd.GeoDataFrame({HEIGHT_FIELD: [r[1] for r in rows]}, geometry=[r[0] for r in rows], crs="EPSG:32723")
    n = int(W / PIX), int(H / PIX)
    from rasterio.features import rasterize
    from rasterio.transform import from_origin
    f = 5
    tr = from_origin(x0, y0 + H, PIX / f, PIX / f)
    m = rasterize(((g, 1) for g in gdf.geometry), out_shape=(n[1] * f, n[0] * f), transform=tr, fill=0, dtype="uint8")
    cover = m.reshape(n[1], f, n[0], f).mean(axis=(1, 3))
    roof = np.array([186, 176, 164]); road = np.array([92, 92, 98]); tree = np.array([86, 120, 78])
    veg = rng.random((n[1], n[0])) < 0.12
    img = (cover[..., None] * roof + (1 - cover[..., None]) * np.where(veg[..., None], tree, road)).astype(float)
    img += rng.normal(0, 7, img.shape)
    rgb = np.clip(img, 0, 255).astype("uint8")
    return rgb, (x0, x0 + W, y0, y0 + H), gdf, (x0 + W / 2, y0 + H / 2)


def load_real():
    import geopandas as gpd, rasterio
    from pyproj import Transformer
    crs_fp = gpd.read_file(FOOTPRINTS, rows=1).crs
    cx, cy = Transformer.from_crs(4326, crs_fp, always_xy=True).transform(*CENTER_LONLAT)
    h = CHIP_HALF_M
    xmin, ymin = np.floor((cx - h) / PIX) * PIX, np.floor((cy - h) / PIX) * PIX
    xmax, ymax = xmin + 2 * h, ymin + 2 * h
    if RGB_TIF is None:
        epsg = crs_fp.to_epsg() or 32723
        print("RGB_TIF is not set. Export the chip with this Earth Engine snippet,\n"
              "download the GeoTIFF from Drive, set RGB_TIF and run again.\n")
        print(f"""// S2 natural colour chip for the feature-scheme figure
var region = ee.Geometry.Rectangle([{xmin:.0f}, {ymin:.0f}, {xmax:.0f}, {ymax:.0f}], 'EPSG:{epsg}', false);
var sr = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(region).filterDate('2019-01-01', '2021-01-01');
var cs = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED').filterBounds(region).filterDate('2019-01-01', '2021-01-01');
var clear = sr.linkCollection(cs, ['cs_cdf']).map(function (i) {{ return i.updateMask(i.select('cs_cdf').gte(0.6)); }});
var rgb = clear.select(['B4', 'B3', 'B2']).median().visualize({{min: 0, max: 3000, gamma: 1.3}});
Export.image.toDrive({{image: rgb, description: 's2_rgb_chip', folder: 'frantz_figures', region: region,
  crs: 'EPSG:{epsg}', crsTransform: [10, 0, {xmin:.0f}, 0, -10, {ymax:.0f}], fileFormat: 'GeoTIFF'}});""")
        return None
    with rasterio.open(RGB_TIF) as src:
        arr = src.read([1, 2, 3])
        b = src.bounds
        rcrs = src.crs
    rgb = np.moveaxis(arr, 0, -1)
    if rgb.dtype != np.uint8:
        lo, hi = np.percentile(rgb, [2, 98]); rgb = (np.clip((rgb - lo) / (hi - lo), 0, 1) ** (1 / 1.2) * 255).astype("uint8")
    gdf = gpd.read_file(FOOTPRINTS, bbox=(xmin - 100, ymin - 100, xmax + 100, ymax + 100))
    if rcrs and gdf.crs != rcrs:
        gdf = gdf.to_crs(rcrs)
        cx, cy = Transformer.from_crs(4326, rcrs, always_xy=True).transform(*CENTER_LONLAT)
    return rgb, (b.left, b.right, b.bottom, b.top), gdf, (cx, cy)


# ------------------- choosing the two example pixels ---------------------
def pick_pair(gdf, ext, ctr):
    c = gdf.geometry.centroid
    gdf = gdf.assign(mx=c.x.to_numpy(), my=c.y.to_numpy(), area=gdf.geometry.area.to_numpy())
    px, py = np.floor(gdf.mx / PIX) * PIX + PIX / 2, np.floor(gdf.my / PIX) * PIX + PIX / 2
    gdf["pcx"], gdf["pcy"] = px, py
    gdf["off"] = np.hypot(gdf.mx - px, gdf.my - py)
    if PICK_XY:
        out = []
        for (x, y) in PICK_XY:
            i = np.argmin(np.hypot(gdf.mx - x, gdf.my - y)); out.append(gdf.iloc[i])
        return gdf, out
    m = 75
    ok = gdf[(gdf.off <= PIX / 2) & (gdf.area >= MIN_AREA_M2) & (gdf.mx > ext[0] + m) & (gdf.mx < ext[1] - m)
             & (gdf.my > ext[2] + m) & (gdf.my < ext[3] - m)]
    if len(ok) < 2:
        raise ValueError("Fewer than two candidate buildings inside the chip; move CENTER_LONLAT or set PICK_XY.")
    a = ok.iloc[np.argmin(np.hypot(ok.mx - (ctr[0] - 35), ok.my - ctr[1]))]
    d = np.hypot(ok.mx - a.mx, ok.my - a.my)
    ok2 = ok[(d >= 55) & (d <= 120)]
    ok2 = ok2 if len(ok2) else ok[d >= 55]
    b = ok2.iloc[np.argmin(np.hypot(ok2.mx - (ctr[0] + 45), ok2.my - ctr[1]))]
    return gdf, [a, b]


def label_of(gdf, pcx, pcy):
    inside = np.hypot(gdf.mx - pcx, gdf.my - pcy) <= R_M
    g = gdf[inside]
    return float(np.average(g[HEIGHT_FIELD], weights=g.area)), int(inside.sum()), inside.to_numpy()


# ------------------- drawing helpers -------------------------------------
def P(u, v, z=0.0, ox=0.0, oy=0.0, s=1.0):
    return ox + (u - v) * C30 * s, oy + (u + v) * S30 * s + z


def iso_sheet(ax, ox, oy, size, z, n, face, edge, zo, mark=None, circle=None):
    pts = [P(0, 0, z, ox, oy), P(size, 0, z, ox, oy), P(size, size, z, ox, oy), P(0, size, z, ox, oy)]
    ax.add_patch(Polygon(pts, closed=True, fc=face, ec=edge, lw=1.2, zorder=zo))
    for k in range(1, n):
        t = size * k / n
        for (p, q) in (((t, 0), (t, size)), ((0, t), (size, t))):
            (x1, y1), (x2, y2) = P(*p, z, ox, oy), P(*q, z, ox, oy)
            ax.plot([x1, x2], [y1, y2], color="white", lw=0.6, alpha=0.7, zorder=zo + 0.1)
    if mark:
        u, v = mark; c = size / n
        if circle:
            u0, v0 = u + c / 2, v + c / 2
            for i in range(n):                    # pixels inside the 50 m (5 px) radius
                for j in range(n):
                    if np.hypot(i - (n // 2), j - (n // 2)) <= circle and (i, j) != (n // 2, n // 2):
                        cell = [P(i * c, j * c, z, ox, oy), P(i * c + c, j * c, z, ox, oy),
                                P(i * c + c, j * c + c, z, ox, oy), P(i * c, j * c + c, z, ox, oy)]
                        ax.add_patch(Polygon(cell, closed=True, fc=BLUE_D, ec="none", alpha=0.35, zorder=zo + 0.2))
            th = np.linspace(0, 2 * np.pi, 120)
            xy = np.array([P(u0 + circle * c * np.cos(t), v0 + circle * c * np.sin(t), z, ox, oy) for t in th])
            ax.plot(xy[:, 0], xy[:, 1], color="black", lw=3.2, alpha=0.35, zorder=zo + 0.4)
            ax.plot(xy[:, 0], xy[:, 1], color="white", lw=1.8, ls=(0, (4, 2.5)), zorder=zo + 0.5)
        cell = [P(u, v, z, ox, oy), P(u + c, v, z, ox, oy), P(u + c, v + c, z, ox, oy), P(u, v + c, z, ox, oy)]
        ax.add_patch(Polygon(cell, closed=True, fc=RED, ec="white", lw=0.8, zorder=zo + 0.6))


def flat_axes(fig, rect):
    ax = fig.add_axes(rect); ax.axis("off")
    w, h = fig.get_size_inches() * np.array(rect[2:])
    ax.set_xlim(0, 10); ax.set_ylim(0, 10 * h / w); ax.set_aspect("equal")
    return ax


def arrow(ov, p, q, **kw):
    ov.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=16, lw=2, color=INK2, shrinkA=0, shrinkB=0, **kw))


def box_text(ax, x, y, text, w=None, fs=9.5, color=INK, weight="normal", ha="left", va="top"):
    return ax.text(x, y, text, fontsize=fs, color=color, weight=weight, ha=ha, va=va, linespacing=1.35)


# ------------------- the figure ------------------------------------------
def make_figure():
    os.makedirs(OUT, exist_ok=True)
    data = demo_data() if DEMO else load_real()
    if data is None:
        print("\nNext: export the chip, set RGB_TIF, run this cell again.")
        return None
    rgb, ext, gdf, ctr = data
    gdf, pair = pick_pair(gdf, ext, ctr)
    info = []
    for s in pair:
        lab, n, inside = label_of(gdf, s.pcx, s.pcy)
        info.append(dict(x=s.pcx, y=s.pcy, cx=s.mx, cy=s.my, label=lab, n=n, inside=inside))
    A, B = info
    print(f"A: label {A['label']:.1f} m from {A['n']} parts | B: label {B['label']:.1f} m from {B['n']} parts | "
          f"distance {np.hypot(A['x'] - B['x'], A['y'] - B['y']):.0f} m")

    plt.rcParams.update({"font.size": 10, "figure.facecolor": SURF, "text.color": INK})
    fig = plt.figure(figsize=(19, 9.6), facecolor=SURF)
    ov = fig.add_axes([0, 0, 1, 1]); ov.set_xlim(0, 1); ov.set_ylim(0, 1); ov.axis("off"); ov.set_zorder(10)

    # ---- panel 1: map --------------------------------------------------
    axm = fig.add_axes([0.275, 0.525, 0.33, 0.40]); axm.set_zorder(1)
    xmin = np.floor((min(A["x"], B["x"]) - 72) / PIX) * PIX; xmax = np.ceil((max(A["x"], B["x"]) + 72) / PIX) * PIX
    ymin = np.floor((min(A["y"], B["y"]) - 68) / PIX) * PIX; ymax = np.ceil((max(A["y"], B["y"]) + 68) / PIX) * PIX
    axm.imshow(rgb, extent=ext, origin="upper", interpolation="nearest", zorder=1)
    for x in np.arange(np.ceil(xmin / PIX) * PIX, xmax + 1, PIX):
        axm.plot([x, x], [ymin, ymax], color="white", lw=0.5, alpha=0.4, zorder=2)
    for y in np.arange(np.ceil(ymin / PIX) * PIX, ymax + 1, PIX):
        axm.plot([xmin, xmax], [y, y], color="white", lw=0.5, alpha=0.4, zorder=2)
    for geom in gdf.geometry:
        if geom.is_empty:
            continue
        for g in (geom.geoms if geom.geom_type == "MultiPolygon" else [geom]):
            x, y = g.exterior.xy
            axm.add_patch(Polygon(np.c_[x, y], closed=True, fc="none", ec="#ffd24a", lw=1.1, zorder=3))
    for s in (A, B):
        for geom in gdf.geometry[s["inside"]]:
            for g in (geom.geoms if geom.geom_type == "MultiPolygon" else [geom]):
                x, y = g.exterior.xy
                axm.add_patch(Polygon(np.c_[x, y], closed=True, fc=ORANGE, ec="#ffd24a", lw=1.1, alpha=0.55, zorder=3.5))
    for s, tag in ((A, "A"), (B, "B")):
        axm.add_patch(Circle((s["x"], s["y"]), R_M, fill=False, ec="black", lw=4.0, alpha=0.45, zorder=4))
        axm.add_patch(Circle((s["x"], s["y"]), R_M, fill=False, ec="white", lw=2.2, ls=(0, (5, 3)), zorder=5))
        axm.add_patch(Rectangle((s["x"] - PIX / 2, s["y"] - PIX / 2), PIX, PIX, fc=RED, ec=RED, lw=2.2, alpha=0.35, zorder=6))
        axm.add_patch(Rectangle((s["x"] - PIX / 2, s["y"] - PIX / 2), PIX, PIX, fill=False, ec=RED, lw=2.4, zorder=6.5))
        axm.plot(s["cx"], s["cy"], "o", ms=6, mfc="black", mec="white", mew=1.4, zorder=7)
        axm.annotate("", xy=(s["x"] + R_M * 0.707, s["y"] + R_M * 0.707), xytext=(s["x"], s["y"]),
                     arrowprops=dict(arrowstyle="-", color="white", lw=1.2), zorder=5)
        axm.text(s["x"] + R_M * 0.42, s["y"] + R_M * 0.42, "50 m", color="white", fontsize=9, weight="bold",
                 ha="left", va="bottom", zorder=8, bbox=dict(fc="black", ec="none", alpha=0.45, pad=1.2))
        axm.text(s["x"] - R_M + 3, s["y"] - R_M + 3, f"{tag}   label {s['label']:.1f} m", color=INK, fontsize=10, weight="bold",
                 ha="left", va="bottom", zorder=8, bbox=dict(fc="white", ec="none", alpha=0.9, pad=2.5))
    axm.set_xlim(xmin, xmax); axm.set_ylim(ymin, ymax); axm.set_aspect("equal", adjustable="box")
    axm.set_xticks([]); axm.set_yticks([])
    for sp in axm.spines.values():
        sp.set_color(INK2); sp.set_linewidth(1.2)
    sx = xmax - 8 - 50; sy = ymin + 7
    axm.add_patch(Rectangle((sx, sy), 50, 4, fc="white", ec="black", lw=1, zorder=9))
    axm.text(sx + 25, sy + 7, "50 m", ha="center", va="bottom", fontsize=9, color="white", weight="bold", zorder=9,
             bbox=dict(fc="black", ec="none", alpha=0.45, pad=1))
    if DEMO:
        axm.text((xmin + xmax) / 2, (ymin + ymax) / 2, "PLACEHOLDER IMAGERY (synthetic)", color="white", alpha=0.55,
                 fontsize=22, weight="bold", ha="center", va="center", rotation=12, zorder=20)
    handles = [Line2D([], [], color="#ffd24a", lw=1.5, label="building part"),
               Line2D([], [], marker="s", ls="", mfc=ORANGE, mec="#ffd24a", alpha=0.7, ms=9, label="parts in the label"),
               Line2D([], [], marker="o", ls="", mfc="black", mec="white", ms=7, label="part centroid"),
               Line2D([], [], marker="s", ls="", mfc=(1, .3, .3, .35), mec=RED, mew=2, ms=10, label="candidate pixel"),
               Line2D([], [], color="#555555", lw=2, ls=(0, (5, 3)), label="50 m radius")]
    leg = axm.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, fontsize=8.2, frameon=False,
                     handlelength=1.6, columnspacing=1.2)
    for t in leg.get_texts():
        t.set_color(INK2)
    ov.text(0.275, 0.965, "1  Sample pixel and its 50 m neighbourhood", fontsize=13, weight="bold", va="center")
    ov.text(0.275, 0.940, "zoom on the red box of panel 0, with 10 m pixels and building footprints", fontsize=10, color=INK2, va="center")
    ov.text(0.025, 0.462, "Label = area-weighted mean height of the parts whose centroid is within 50 m of the pixel centre (Frantz et al. 2021).\n"
            "Candidate pixel: a part centroid lies within 5 m of its centre. Samples are >= 50 m apart, so circles may overlap (hence spatial blocks).", fontsize=8.8, color=INK2, va="top", linespacing=1.35)

    # ---- panel 2: stack ------------------------------------------------
    axs = flat_axes(fig, [0.65, 0.50, 0.34, 0.43])
    ov.text(0.65, 0.965, "2  Layer stack at the same 10 m pixel", fontsize=13, weight="bold", va="center")
    ov.text(0.65, 0.940, "all clear observations of the 2-year window (Cloud Score+ >= 0.6)", fontsize=10, color=INK2, va="center")
    size, ox, oy = 3.3, 2.0, 0.35
    tints = ["#cfe0f5", "#b6d0ee", "#9dc0e6", "#84b0de"]
    mark = (1.45, 1.45)
    for k, (z, tc) in enumerate(zip([0.0, 0.85, 1.7, 2.55], tints)):
        iso_sheet(axs, ox, oy, size, z, 13, tc, BLUE_D, 2 + k, mark=(size / 13 * 6, size / 13 * 6), circle=5)
    top = P(size / 13 * 6.5, size / 13 * 6.5, 0.0, ox, oy)
    axs.plot([top[0], top[0]], [top[1] - 0.1, top[1] + 2.55 + 0.1], color=RED, lw=1.6, ls=(0, (3, 2)), zorder=12)
    axs.text(ox + 1.25, oy + 2.55 + 2.55, "50 m radius\n(5 px)", fontsize=8.8, color=INK, ha="left", va="bottom", weight="bold", zorder=14)
    axs.text(2.3, 0.15, "every layer is a 10 m raster; shaded pixels = inside the 50 m radius", ha="left", fontsize=9, color=INK2)
    axs.text(top[0] + 0.15, oy - 0.45, "same pixel in every layer", fontsize=9, color=RED, ha="left", va="top")
    axs.text(ox - 0.5, oy + 1.6, "...", fontsize=22, ha="center", color=INK2)
    # notes (not drawn as matrices)
    tx = 5.35
    box_text(axs, tx, 6.0, "Layers (18)", weight="bold", fs=10.5)
    box_text(axs, tx, 5.55, "Sentinel-1 (2): VV, VH\nSentinel-2 (10): B2 B3 B4 B5 B6\n    B7 B8 B8A B11 B12\nIndices (6): NDVI, TCG, NDBI,\n    TCB, TCW, mNDWI", fs=9.5)
    box_text(axs, tx, 3.35, "Time", weight="bold", fs=10.5)
    box_text(axs, tx, 2.9, "2 years from 2019 (2019 to 2020),\nall cloud-free observations.\nThe tabular branch summarises the\nwhole window; the CNN branch uses\nseasonal composites of it.", fs=9.5)
    axs.set_ylim(0, 6.6)

    # ---- panel 3a: tabular ---------------------------------------------
    axt = flat_axes(fig, [0.075, 0.03, 0.379, 0.35])
    box_text(axt, 0.0, axt.get_ylim()[1], "3a  Structured data for classical ML", weight="bold", fs=13, va="top")
    box_text(axt, 0.0, axt.get_ylim()[1] - 0.45, "one row of 1,872 features per sample pixel", fs=10, color=INK2)
    top_y = axt.get_ylim()[1] - 1.15
    # step: statistics
    axt.add_patch(Rectangle((0.0, top_y - 2.55), 3.0, 2.55, fc="white", ec=GRID, lw=1.2))
    box_text(axt, 0.15, top_y - 0.12, "13 temporal statistics\nper layer", weight="bold", fs=9.5)
    box_text(axt, 0.15, top_y - 0.82, "mean, stdDev, p0, p10, p25,\np50, p75, p90, p100, range,\nIQR, skew, kurtosis", fs=9)
    box_text(axt, 0.15, top_y - 1.95, "18 layers x 13 = 234 rasters", fs=9.5, weight="bold", color=BLUE_D)
    # step: morphology (kernel graphic)
    kx, ky, cs = 3.55, top_y - 1.65, 0.20
    n = 13; c0 = n // 2
    axt.add_patch(Rectangle((3.45, top_y - 2.55), 3.25, 2.55, fc="white", ec=GRID, lw=1.2))
    for i in range(n):
        for j in range(n):
            inside = np.hypot(i - c0, j - c0) <= 5.0
            fc = "#cfe0f5" if inside else "white"
            if i == c0 and j == c0:
                fc = RED
            axt.add_patch(Rectangle((kx + 0.0 + i * cs * 0.55 + 0.0, ky + j * cs * 0.55 - 0.05 + 0.5 * 0 ), cs * 0.55, cs * 0.55, fc=fc, ec="#c9c8c2", lw=0.3))
    box_text(axt, 3.6, top_y - 0.12, "7 morphological operators", weight="bold", fs=9.5)
    box_text(axt, 5.05, top_y - 0.55, "ERO  DIL  OPN\nCLS  GRD  THT\nBHT  + BASE", fs=8.8)
    box_text(axt, 5.05, top_y - 1.45, "circular 50 m\n(5 px) element", fs=8.8, color=INK2)
    box_text(axt, 3.6, top_y - 2.25, "234 x 8 = 1,872 features", fs=9.5, weight="bold", color=BLUE_D)
    # step: table
    tx0, ty0 = 7.15, top_y - 0.15
    axt.add_patch(Rectangle((7.0, top_y - 2.55), 3.0, 2.55, fc="white", ec=GRID, lw=1.2))
    cols = ["f1", "f2", "...", "f1872", "height"]
    cw = [0.5, 0.5, 0.45, 0.78, 0.62]
    xx = tx0
    for c, w in zip(cols, cw):
        axt.add_patch(Rectangle((xx, ty0 - 0.38), w, 0.38, fc="#eceae4" if c != "height" else "#fbe2d7", ec=GRID, lw=0.8))
        axt.text(xx + w / 2, ty0 - 0.19, c, ha="center", va="center", fontsize=8.2); xx += w
    for r in range(4):
        xx = tx0
        for c, w in zip(cols, cw):
            hl = r == 1
            axt.add_patch(Rectangle((xx, ty0 - 0.38 * (r + 2)), w, 0.38, fc="#f9d3cf" if hl else "white", ec=GRID, lw=0.8))
            xx += w
    axt.text(tx0 + sum(cw) / 2 - 0.1, ty0 - 0.38 * 3 + 0.19, "pixel A", fontsize=8.2, color=RED, ha="center", va="center")
    box_text(axt, 7.1, ty0 - 0.38 * 5 - 0.08, "1 row per sample, label = height", fs=8.3, color=INK2)
    for x1, x2 in ((3.0, 3.45), (6.7, 7.0)):
        axt.add_patch(FancyArrowPatch((x1, top_y - 1.27), (x2, top_y - 1.27), arrowstyle="-|>", mutation_scale=13, lw=1.6, color=INK2))
    box_text(axt, 0.0, top_y - 2.9, "-> SVR, HistGB, random forest, KNN, MLP, Ridge, stacking", fs=9.5, color=INK2)

    # ---- panel 3b: CNN cube ----------------------------------------------
    axc = flat_axes(fig, [0.545, 0.03, 0.37, 0.35])
    box_text(axc, 0.0, axc.get_ylim()[1], "3b  Unstructured data for a CNN", weight="bold", fs=13, va="top")
    box_text(axc, 0.0, axc.get_ylim()[1] - 0.45, "one image patch (data cube) per sample pixel", fs=10, color=INK2)
    top_y = axc.get_ylim()[1] - 1.15
    axc.add_patch(Rectangle((0.0, top_y - 2.55), 3.3, 2.55, fc="white", ec=GRID, lw=1.2))
    box_text(axc, 0.15, top_y - 0.12, "4 seasonal medians", weight="bold", fs=9.5)
    box_text(axc, 0.15, top_y - 0.58, "Jan-Mar, Apr-Jun, Jul-Sep,\nOct-Dec, pooled over the\n2-year window", fs=9)
    box_text(axc, 0.15, top_y - 1.55, "x (VV, VH + 10 S2 bands)\n= 48 channels", fs=9.5, weight="bold", color=BLUE_D)
    box_text(axc, 0.15, top_y - 2.2, "no indices, no morphology", fs=8.6, color=INK2)
    # cube
    cs_, cx0, cy0, hz = 1.5, 4.9, top_y - 2.5, 1.3
    s = cs_
    top = [P(0, 0, hz, cx0, cy0, 1), P(s, 0, hz, cx0, cy0, 1), P(s, s, hz, cx0, cy0, 1), P(0, s, hz, cx0, cy0, 1)]
    left = [P(0, 0, 0, cx0, cy0, 1), P(0, s, 0, cx0, cy0, 1), P(0, s, hz, cx0, cy0, 1), P(0, 0, hz, cx0, cy0, 1)]
    right = [P(0, 0, 0, cx0, cy0, 1), P(s, 0, 0, cx0, cy0, 1), P(s, 0, hz, cx0, cy0, 1), P(0, 0, hz, cx0, cy0, 1)]
    axc.add_patch(Polygon(left, closed=True, fc="#9dc0e6", ec=BLUE_D, lw=1.2, zorder=2))
    axc.add_patch(Polygon(right, closed=True, fc="#b6d0ee", ec=BLUE_D, lw=1.2, zorder=2))
    axc.add_patch(Polygon(top, closed=True, fc="#e6eefa", ec=BLUE_D, lw=1.2, zorder=3))
    for k in range(1, 12):                       # channel stripes on the front faces
        z = hz * k / 12
        for (u, v, du, dv) in ((0, 0, 0, s), (0, 0, s, 0)):
            (x1, y1), (x2, y2) = P(u, v, z, cx0, cy0, 1), P(u + du, v + dv, z, cx0, cy0, 1)
            axc.plot([x1, x2], [y1, y2], color="white", lw=0.7, alpha=0.8, zorder=2.5)
    for k in range(1, 11):                       # 33 x 33 grid hinted on the top face
        t = s * k / 11
        for (p, q) in (((t, 0), (t, s)), ((0, t), (s, t))):
            (x1, y1), (x2, y2) = P(*p, hz, cx0, cy0, 1), P(*q, hz, cx0, cy0, 1)
            axc.plot([x1, x2], [y1, y2], color="#c7d6ea", lw=0.5, zorder=3.2)
    a, b_ = s * 11 / 33 * 1.0, s * 22 / 33 * 1.0   # 11 x 11 centre crop of the 33 x 33 patch
    crop = [P(a, a, hz, cx0, cy0, 1), P(b_, a, hz, cx0, cy0, 1), P(b_, b_, hz, cx0, cy0, 1), P(a, b_, hz, cx0, cy0, 1)]
    axc.add_patch(Polygon(crop, closed=True, fc=(1, .3, .3, .25), ec=RED, lw=2, zorder=4))
    centre = P(s / 2, s / 2, hz, cx0, cy0, 1)
    axc.plot(*centre, "s", ms=4, color=RED, zorder=5)
    p0 = P(0, 0, 0, cx0, cy0, 1)
    axc.text(cx0 - 0.9, cy0 + 0.05, "33 px", fontsize=8.8, ha="right", va="top")
    axc.text(cx0 + 0.9, cy0 + 0.05, "33 px", fontsize=8.8, ha="left", va="top")
    axc.text(cx0 + s * C30 + 0.12, cy0 + hz / 2 + s * S30 * 0.0 + 0.5, "48\nchannels", fontsize=9, ha="left", va="center", color=BLUE_D)
    box_text(axc, 7.45, top_y - 0.1, "patch (330 m) centred on\nthe sample pixel", fs=9)
    box_text(axc, 7.45, top_y - 0.95, "red square: optional\n11 x 11 px (110 m) crop,\nthe support of Frantz's\n50 m kernel", fs=8.8, color=INK2)
    box_text(axc, 7.45, top_y - 2.25, "-> small CNN", fs=9.5, color=INK2)
    axc.add_patch(FancyArrowPatch((3.3, top_y - 1.27), (3.62, top_y - 1.27), arrowstyle="-|>", mutation_scale=13, lw=1.6, color=INK2))

    # ---- arrows between panels (overlay) -----------------------------------
    arrow(ov, (0.612, 0.725), (0.645, 0.725))
    ov.text(0.628, 0.745, "pixel A", ha="center", fontsize=9, color=INK2)
    ov.plot([0.718, 0.718], [0.52, 0.408], color=INK2, lw=2)
    ov.plot([0.276, 0.718], [0.408, 0.408], color=INK2, lw=2)
    arrow(ov, (0.276, 0.408), (0.276, 0.383))
    arrow(ov, (0.718, 0.408), (0.718, 0.383))
    ov.text(0.286, 0.396, "13 statistics + morphology", fontsize=9, color=INK2, ha="left", va="center")
    ov.text(0.728, 0.396, "4 seasonal medians", fontsize=9, color=INK2, ha="left", va="center")
    # ---- panel 0: the whole raw chip with the zoom window --------------------
    ax0 = fig.add_axes([0.025, 0.525, 0.19, 0.40]); ax0.set_zorder(1)
    ax0.imshow(rgb, extent=ext, origin="upper", interpolation="nearest")
    ax0.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, ec="black", lw=4.5, alpha=0.4, zorder=5))
    ax0.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, ec=RED, lw=2.4, zorder=6))
    ax0.set_xlim(ext[0], ext[1]); ax0.set_ylim(ext[2], ext[3]); ax0.set_aspect("equal", adjustable="box")
    ax0.set_xticks([]); ax0.set_yticks([])
    for sp in ax0.spines.values():
        sp.set_color(INK2); sp.set_linewidth(1.2)
    sb = ext[1] - 12 - 100
    ax0.add_patch(Rectangle((sb, ext[2] + 10), 100, 6, fc="white", ec="black", lw=1, zorder=9))
    ax0.text(sb + 50, ext[2] + 20, "100 m", ha="center", va="bottom", fontsize=8.5, color="white", weight="bold", zorder=9,
             bbox=dict(fc="black", ec="none", alpha=0.45, pad=1))
    if DEMO:
        ax0.text((ext[0] + ext[1]) / 2, (ext[2] + ext[3]) / 2, "PLACEHOLDER", color="white", alpha=0.55, fontsize=13, weight="bold",
                 ha="center", va="center", rotation=12, zorder=20)
    ov.text(0.025, 0.965, "0  Raw imagery", fontsize=13, weight="bold", va="center")
    ov.text(0.025, 0.940, f"Sentinel-2 natural colour, 10 m, {int(round(ext[1] - ext[0]))} x {int(round(ext[3] - ext[2]))} m chip", fontsize=10, color=INK2, va="center")
    fig.canvas.draw()
    inv = fig.transFigure.inverted()
    p_tr = inv.transform(ax0.transData.transform((xmax, ymax))); p_br = inv.transform(ax0.transData.transform((xmax, ymin)))
    q_tl = inv.transform(axm.transAxes.transform((0, 1))); q_bl = inv.transform(axm.transAxes.transform((0, 0)))
    for p, q in ((p_tr, q_tl), (p_br, q_bl)):
        ov.plot([p[0], q[0]], [p[1], q[1]], color=RED, lw=1.4, alpha=0.9, zorder=1)
    ov.text(0.03, 0.012, "Features follow Frantz et al. (2021): morphological operators on spectral-temporal metrics (not GLCM texture). "
            "Earth Engine processing replaces FORCE; window and cities are ours.", fontsize=8.5, color=INK2, va="bottom")
    for fmt in ("png", "pdf"):
        fig.savefig(f"{OUT}/feature_extraction_scheme{'_DEMO' if DEMO else ''}.{fmt}", dpi=200, facecolor=SURF)
    plt.show()
    return fig


fig = make_figure()